# Gemma 4 AML Fine-Tune -- GGUF -- HuggingFace Hub

**Pipeline:**
1. Install Unsloth + deps
2. QLoRA fine-tune on `framl_train_combined_v26.jsonl` (713 examples)
3. Merge LoRA -> export F16 GGUF -> quantise Q4_K_M
4. Upload GGUF to HuggingFace Hub
5. Register with Ollama as `aria-v2`

> **NOTE:** Gemma 4 model IDs and chat template tokens should be verified
> against the Unsloth release notes before running -- marked with `# VERIFY` comments.


## Cell 1 -- Install Dependencies
Unsloth must be installed before transformers/trl.

In [ ]:
!/venv/main/bin/pip install unsloth unsloth_zoo transformers trl peft datasets \
    bitsandbytes accelerate sentence-transformers huggingface_hub

# torchcodec pulls in FFmpeg 4.x dependency that conflicts with vast.ai system FFmpeg 5+
!/venv/main/bin/pip uninstall -y torchcodec 2>/dev/null || true

# Flash Attention 2 -- try precompiled wheel only (--only-binary=:all: makes pip fail
# instantly if no wheel exists rather than hanging on a 20-min source build).
# Unsloth has its own built-in attention kernels so training works fine without it.
import subprocess
fa_result = subprocess.run(
    ["/venv/main/bin/pip", "install", "flash-attn",
     "--only-binary=:all:", "--no-deps", "--quiet"],
    capture_output=True, text=True
)
if fa_result.returncode == 0:
    print("flash-attn: installed")
else:
    print("flash-attn: no precompiled wheel for this torch/CUDA version -- skipping")
    print("  Unsloth built-in kernels are active, training unaffected.")

import unsloth; print("Unsloth:", unsloth.__version__)

## Cell 2 -- Environment & Imports
`HF_HOME` must be set **before** any HuggingFace import.

In [ ]:
import os
os.environ['HF_HOME'] = '/dev/shm/huggingface'   # RAM-backed tmpfs -- fast, ~15GB free, wiped on stop
os.environ['HF_TOKEN'] = ''  # paste your token here

import torch
from pathlib import Path
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

print(f'CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 3 -- Configuration

**GPU guide:**
- RTX 3090 / 4090 (24 GB) -> `gemma-4-4b-it` (E4B), batch=2, accum=4
- A100 40 GB -> `gemma-4-27b-it`, batch=1, accum=8


In [ ]:
# -- Model --
BASE_MODEL  = 'unsloth/gemma-4-E4B-it'   # VERIFY: check Unsloth hub for exact ID
MAX_SEQ_LEN = 4096

# -- LoRA --
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                  'gate_proj', 'up_proj', 'down_proj']

# -- Training --
EPOCHS     = 3        # 956 examples x 3 epochs
BATCH_SIZE = 2
GRAD_ACCUM = 4        # effective batch = 8
LR         = 2e-4

# -- Paths --
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'finetune':
    REPO_ROOT = REPO_ROOT.parent
DATA_FILE  = REPO_ROOT / 'finetune' / 'data' / 'aria_train_combined_v39_full.jsonl'
OUT_DIR    = Path('/workspace/aria-v2-adapter')
GGUF_DIR   = Path('/workspace/aria-v2-gguf')
Q8_GGUF    = Path('/workspace/aria-v2-q8.gguf')
Q4KM_GGUF  = Path('/workspace/aria-v2-q4km.gguf')
OUT_DIR.mkdir(parents=True, exist_ok=True)
GGUF_DIR.mkdir(parents=True, exist_ok=True)

# -- HuggingFace Hub --
HF_REPO  = 'speri420/aria-v2'
HF_TOKEN = os.environ.get('HF_TOKEN', '')

# -- Ollama --
OLLAMA_MODEL_NAME = 'aria-v2'

print(f'Base model: {BASE_MODEL}')
print(f'Data:       {DATA_FILE}  (exists={DATA_FILE.exists()})')
print(f'GGUF out:   {Q4KM_GGUF}')
print(f'HF repo:    {HF_REPO}  (token set={bool(HF_TOKEN)})')

## Cell 4 -- Load Base Model (4-bit QLoRA)

In [ ]:
print(f'Loading {BASE_MODEL} in 4-bit QLoRA ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
print('Model loaded.')
tmpl = tokenizer.chat_template or 'NOT SET'
print(f'Chat template (first 120): {tmpl[:120]}')

## Cell 5 -- Attach LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = TARGET_MODULES,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
model.print_trainable_parameters()


## Cell 6 -- Load & Format Dataset

Gemma 4 uses `<start_of_turn>` / `<end_of_turn>` tokens.
`tokenizer.apply_chat_template()` handles the format automatically.

> **VERIFY:** If tool_calls turns cause errors, the Unsloth Gemma 4 tokenizer
> may need tool role handling -- check Unsloth release notes.


In [ ]:
import json

def load_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def format_example(example):
    msgs = example['messages']
    # Pass messages through unchanged -- apply_chat_template handles tool_calls
    # and role:tool natively, producing the correct Gemma 4 format:
    #   <|tool_call>call:name{"arg": "val"}<tool_call|>   (tool call)
    #   <|turn>tool{content}<turn|>                      (tool result)
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {'text': text}

raw = load_jsonl(str(DATA_FILE))
print(f'Loaded {len(raw)} examples from {DATA_FILE.name}')
dataset = Dataset.from_list(raw).map(format_example, remove_columns=['messages'])
print(f'Formatted: {len(dataset)} rows')
print('Sample (first 800 chars):')
print(dataset[0]['text'][:800])


## Cell 7 -- Train

In [ ]:
sft_config = SFTConfig(
    output_dir                  = str(OUT_DIR),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.05,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 10,
    save_strategy               = 'epoch',
    optim                       = 'adamw_8bit',
    dataset_text_field          = 'text',
    max_seq_length              = MAX_SEQ_LEN,
    dataset_num_proc            = 2,
    report_to                   = 'none',
)
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args          = sft_config,
)

# Mask prompt tokens from loss -- only model response tokens contribute.
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part    = "<|turn>model\n",
)
print('train_on_responses_only applied.')
print('Run Cell 7b next to verify masking before training starts.')

## Cell 7b -- Verify Masking + Train

Checks that `train_on_responses_only` is actually masking instruction tokens before
committing to a full training run. If masking is broken it prints the raw tokenized
format so you can find the correct delimiter and fix Cell 7.

In [ ]:
# ── Verify masking before committing to a full training run ──────────────────
_dl = trainer.get_train_dataloader()
_batch = next(iter(_dl))
_labels = _batch['labels']

n_masked  = (_labels == -100).sum().item()
n_total   = _labels.numel()
pct_masked = 100 * n_masked / n_total
print(f"Masking check: {n_masked:,} / {n_total:,} tokens masked = {pct_masked:.1f}%")

if pct_masked < 20:
    # Show the raw formatted text so we can identify the correct delimiter
    _tok = tokenizer.tokenizer if hasattr(tokenizer, 'tokenizer') else tokenizer
    print("\nFAIL: less than 20% masked -- train_on_responses_only is NOT working.")
    print("Raw formatted example (first 800 chars) -- find the actual turn delimiter:")
    print(repr(dataset[0]['text'][:800]))
    print("\nAlso showing tokenized IDs around the first 60 tokens:")
    _ids = _tok(dataset[0]['text'], return_tensors='pt').input_ids[0]
    for i in range(min(60, len(_ids))):
        tid = _ids[i].item()
        tok = _tok.decode([tid])
        print(f"  [{i:3d}] id={tid:6d}  {repr(tok)}")
    raise RuntimeError(
        "Masking failed. Fix instruction_part/response_part in Cell 7 to match "
        "the actual token format shown above, then re-run Cells 6-7 before training."
    )

print(f"OK: {pct_masked:.0f}% of tokens masked -- masking is working.")
print("Starting training ...\n")

# ── Train ─────────────────────────────────────────────────────────────────────
print(f'Training: {len(dataset)} examples | {EPOCHS} epochs | effective batch={BATCH_SIZE * GRAD_ACCUM}')
trainer.train()
print('Training complete.')
trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))
print(f'Adapter saved to {OUT_DIR}')

In [ ]:
# ── Post-training cleanup -- run before Cell 8b-pre ──────────────────────────
# Training leaves optimizer states and gradient buffers in VRAM (~8-10 GB).
# merge_and_unload() needs headroom -- without this cleanup the kernel crashes.
import torch, gc

del trainer
torch.cuda.empty_cache()
gc.collect()
vram_gb = torch.cuda.memory_allocated() / 1e9
print(f"VRAM after cleanup: {vram_gb:.1f} GB  (should be ~10-11 GB, base model only)")
print("Safe to run Cell 8b-pre.")

## Cell 8a -- Build llama.cpp (skip if already built)

In [ ]:
import subprocess, shutil, sys

CUDA_ARCH = '86'  # RTX 3090/3090Ti=86; RTX 4090=89 -- verify with nvidia-smi

# Use current HEAD -- Gemma 4 (general.architecture=gemma4) is supported by Ollama 0.23.0+.
# aria-v1 (working) also uses gemma4 arch -- the old pin to 223373742 was wrong
# (that commit predates Gemma4ForConditionalGeneration support entirely).
LLAMA_CPP_COMMIT = ""  # empty = use current HEAD

llama_q = Path('/workspace/llama.cpp/build/bin/llama-quantize')
if llama_q.exists():
    rev = subprocess.run('cd /workspace/llama.cpp && git rev-parse --short HEAD',
                         shell=True, capture_output=True, text=True)
    print(f'llama.cpp already built -- skipping.  commit={rev.stdout.strip()}')
else:
    cmds = [
        'apt-get install -y cmake build-essential 2>/dev/null',
        'cd /workspace && git clone https://github.com/ggerganov/llama.cpp || true',
    ]
    if LLAMA_CPP_COMMIT:
        cmds.append(f'cd /workspace/llama.cpp && git checkout {LLAMA_CPP_COMMIT}')
    cmds.append(
        f'cd /workspace/llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH} && cmake --build build --config Release -j$(nproc)'
    )
    for cmd in cmds:
        print(f'>> {cmd[:80]}')
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print('STDERR:', r.stderr[-500:])
            raise RuntimeError(f'Failed: {cmd}')
    rev = subprocess.run('cd /workspace/llama.cpp && git rev-parse --short HEAD',
                         shell=True, capture_output=True, text=True)
    print(f'llama.cpp build complete.  commit={rev.stdout.strip()}')

LLAMA_CONVERT  = Path('/workspace/llama.cpp/convert_hf_to_gguf.py')
LLAMA_QUANTIZE = Path('/workspace/llama.cpp/build/bin/llama-quantize')
print(f'convert:  {LLAMA_CONVERT.exists()}')
print(f'quantize: {LLAMA_QUANTIZE.exists()}')

## Cell 8b-pre -- Merge LoRA + Save BF16 Safetensors

Merges LoRA, dequantizes any remaining 4-bit modules, extracts state dict to CPU,
saves clean BF16 safetensors to `GGUF_DIR`, strips `quantization_config` from
`config.json`, then frees GPU/CPU memory.

**Run before Cell 8b.** Cell 8b only runs the conversion subprocess.

## Cell 8b -- Convert Safetensors -> Q8_0 GGUF

In [ ]:
import gc, json as _json, shutil as _shutil, sys, torch
from pathlib import Path
import safetensors.torch as _st

# ── FORCE_RETRAIN: wipe stale GGUF outputs so conversion re-runs ─────────────
FORCE_RETRAIN = True
if FORCE_RETRAIN:
    for _p in [GGUF_DIR, Q8_GGUF, Q4KM_GGUF]:
        _p = Path(str(_p))
        if _p.is_dir():
            print(f'Removing stale dir {_p}')
            _shutil.rmtree(str(_p))
        elif _p.exists():
            print(f'Removing stale file {_p}')
            _p.unlink()

# ── Step 1: Merge diagnostic ──────────────────────────────────────────────────
import unsloth as _ul
print(f'Unsloth: {_ul.__version__}')

FastLanguageModel.for_inference(model)
_tok = tokenizer.tokenizer if hasattr(tokenizer, 'tokenizer') else tokenizer
_prompt = _tok.apply_chat_template(
    [{"role": "user", "content": "What is your name?"}],
    tokenize=False, add_generation_prompt=True
)
_ids = _tok(_prompt, return_tensors="pt").input_ids.to("cuda")
with torch.no_grad():
    _out = model.generate(input_ids=_ids, max_new_tokens=30, use_cache=True)
print("Unmerged:", _tok.decode(_out[0][_ids.shape[1]:], skip_special_tokens=False))

print("\nMerging LoRA ...")
merged = model.merge_and_unload()

for _name, _p in merged.named_parameters():
    if 'embed_tokens' in _name and _p.numel() > 1000:
        print(f"{_name}: dtype={_p.dtype}, min={_p.min():.6f}, max={_p.max():.6f}")
        assert _p.abs().max() > 1e-4, "ZERO WEIGHTS -- merge broken. Do NOT proceed to Cell 8b."
        break

with torch.no_grad():
    _out2 = merged.generate(input_ids=_ids, max_new_tokens=30, use_cache=True)
print("Merged: ", _tok.decode(_out2[0][_ids.shape[1]:], skip_special_tokens=False))
print("Merge verified -- non-zero weights confirmed.")

# ── Step 2: Explicit bnb dequantization ──────────────────────────────────────
# merge_and_unload() converts LoRA-targeted layers to BF16 but leaves other
# Linear4bit modules with bitsandbytes metadata tensors (.absmax, .quant_map,
# .quant_state, .nested_absmax) still in the state dict.
# convert_hf_to_gguf.py chokes on those metadata tensors -- dequant them first.
try:
    import bitsandbytes as bnb
    n_dq = 0
    for _mod_name, _mod in merged.named_modules():
        if isinstance(_mod, bnb.nn.Linear4bit):
            with torch.no_grad():
                _w = bnb.functional.dequantize_4bit(
                    _mod.weight.data,
                    _mod.weight.quant_state,
                    quant_type=_mod.weight.quant_type,
                ).to(torch.bfloat16)
                _mod.weight = torch.nn.Parameter(_w, requires_grad=False)
            n_dq += 1
    print(f"Dequantized {n_dq} Linear4bit modules to BF16.")
except Exception as _e:
    print(f"bnb dequant skipped ({_e}) -- assuming weights already BF16")

# ── Step 3: Extract CPU state dict, clone tied lm_head ───────────────────────
# Gemma ties embed_tokens.weight and lm_head.weight (shared GPU storage).
# save_file() requires all tensors to be independent -- .clone() on CPU breaks it.
# .detach().cpu() first to avoid GPU OOM (embedding is ~5 GB).
print("Extracting state dict to CPU ...")
sd = {k: v.detach().cpu() for k, v in merged.state_dict().items()}

for _k in list(sd.keys()):
    if 'lm_head' in _k and 'weight' in _k:
        sd[_k] = sd[_k].clone()
        print(f"  Cloned tied weight: {_k}")

total_gb = sum(t.numel() * t.element_size() for t in sd.values()) / 1e9
max_abs = max(t.abs().max().item() for t in sd.values() if t.numel() > 100)
print(f"Weights: {total_gb:.1f} GB  |  max_abs={max_abs:.4f}  (should be > 0.01)")
assert max_abs > 0.01, "State dict is near-zero! Merge or dequant failed."

# ── Step 4: Save safetensors to GGUF_DIR ─────────────────────────────────────
GGUF_DIR.mkdir(parents=True, exist_ok=True)
_out_sf = GGUF_DIR / "model.safetensors"
print(f"Saving {len(sd)} tensors to {_out_sf} ...")
_st.save_file(sd, str(_out_sf))
print(f"Saved: {_out_sf.stat().st_size / 1e9:.1f} GB")
del sd

# ── Step 5: Save config + strip quantization_config ──────────────────────────
# If quantization_config is present in config.json, convert_hf_to_gguf.py
# tries to dequantize from bnb format and raises NotImplementedError. Strip it.
merged.config.save_pretrained(str(GGUF_DIR))
_cfg_path = GGUF_DIR / 'config.json'
_cfg = _json.loads(_cfg_path.read_text())
if _cfg.pop('quantization_config', None) is not None:
    _cfg_path.write_text(_json.dumps(_cfg, indent=2))
    print("Stripped quantization_config from config.json")

_tok.save_pretrained(str(GGUF_DIR))
print(f"Config + tokenizer saved to {GGUF_DIR}")

# ── Step 6: Verify saved file is non-zero ────────────────────────────────────
_chk = _st.load_file(str(_out_sf))
_samp = next((v for k, v in _chk.items() if v.numel() > 1000), None)
assert _samp is not None and _samp.abs().max() > 1e-4, "SAVED FILE IS ZERO-FILLED -- check disk space!"
print(f"Saved safetensors verified non-zero: max_abs={_samp.abs().max():.4f}")
del _chk, _samp

# ── Step 7: Free GPU + CPU memory before conversion ──────────────────────────
# Conversion subprocess loads ~14 GB safetensors into its own process.
# If model (~11 GB VRAM) + state dict RAM remain, total memory = OOM + kernel crash.
del merged, model
torch.cuda.empty_cache()
gc.collect()
vram_mb = torch.cuda.memory_allocated() / 1e6
print(f"GPU freed: {vram_mb:.0f} MB allocated (should be ~0)")
print(f"\nReady for Cell 8b -- GGUF conversion.")

In [ ]:
import subprocess, sys, shutil as _shutil
from pathlib import Path

# ── Convert GGUF_DIR safetensors -> Q8_0 GGUF (~8 GB) ────────────────────────
# GGUF_DIR must contain clean BF16 safetensors (written by Cell 8b-pre).
# --outtype q8_0 converts directly to Q8_0, skipping the F16 intermediate
# and saving ~14 GB of disk space.
if not Q8_GGUF.exists():
    print(f"Converting {GGUF_DIR} -> {Q8_GGUF} ...")
    subprocess.run('df -h /workspace', shell=True)
    r = subprocess.run(
        [sys.executable, str(LLAMA_CONVERT), str(GGUF_DIR),
         '--outfile', str(Q8_GGUF), '--outtype', 'q8_0'],
        capture_output=True, text=True
    )
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1500:])
        raise RuntimeError('q8_0 conversion failed')
    print(f'q8_0 GGUF: {Q8_GGUF.stat().st_size / 1e9:.1f} GB')
    print('Removing merged safetensors to free disk ...')
    _shutil.rmtree(str(GGUF_DIR))
    subprocess.run('df -h /workspace', shell=True)
else:
    print(f'q8_0 exists ({Q8_GGUF.stat().st_size / 1e9:.1f} GB) -- skipping.')

# Verify GGUF magic bytes
with open(str(Q8_GGUF), 'rb') as _f:
    _magic = _f.read(4)
print(f"GGUF magic: {_magic}  (expected: b'GGUF')")
assert _magic == b'GGUF', f"Bad GGUF magic: {_magic}"
print("Cell 8b complete -- Q8_0 GGUF ready. Proceed to Cell 10 (Ollama) or HF upload.")

## Do q4 quantization if needed

In [ ]:

# -- Step 3: Quantise q8_0 -> Q4_K_M (~2.5 GB) --------------------------------
if not Q4KM_GGUF.exists():
    print('Quantising to Q4_K_M ...')
    r = subprocess.run(
        [str(LLAMA_QUANTIZE), str(Q8_GGUF), str(Q4KM_GGUF), 'Q4_K_M', '--nthread', '1'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(r.stderr[-1500:]); raise RuntimeError('Q4_K_M quantisation failed')
    print(f'Q4_K_M: {Q4KM_GGUF.stat().st_size / 1e9:.1f} GB')
    print('Removing q8_0 to free disk ...')
    Q8_GGUF.unlink()
    subprocess.run('df -h /workspace', shell=True)
else:
    print(f'Q4_K_M exists ({Q4KM_GGUF.stat().st_size / 1e9:.1f} GB) -- skipping.')

## Upload GGUF to HuggingFace Hub

Upload the q8 or q4 files to huggingface hub

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path

# Upload whichever GGUF exists -- q8_0 if Q4_K_M not yet produced
Q8_GGUF  = Path('/workspace/aria-v2-q8.gguf')
UPLOAD_FILE = Q4KM_GGUF if Q4KM_GGUF.exists() else Q8_GGUF
if not UPLOAD_FILE.exists():
    raise FileNotFoundError('No GGUF file found -- run Cell 8b or 8c first.')

api = HfApi(token=HF_TOKEN)
try:
    api.create_repo(repo_id=HF_REPO, repo_type='model', exist_ok=True)
    print(f'Repo ready: {HF_REPO}')
except Exception as e:
    print(f'Repo: {e}')

print(f'Uploading {UPLOAD_FILE.name} ({UPLOAD_FILE.stat().st_size / 1e9:.1f} GB) ...')
api.upload_file(
    path_or_fileobj = str(UPLOAD_FILE),
    path_in_repo    = UPLOAD_FILE.name,
    repo_id         = HF_REPO,
    repo_type       = 'model',
)
print(f'Done: https://huggingface.co/{HF_REPO}')
print(f'File uploaded: {UPLOAD_FILE.name}')


## Cell 10 -- Install Ollama (skip if already installed)

In [ ]:
import subprocess, shutil

if shutil.which("ollama"):
    print("Ollama already installed:", shutil.which("ollama"))
else:
    print("Installing Ollama ...")
    r = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, capture_output=True, text=True
    )
    print(r.stdout[-500:])
    if r.returncode != 0:
        print(r.stderr[-500:])
        raise RuntimeError("Ollama install failed")
    print("Ollama installed.")

r = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
print(r.stdout.strip())


## Cell 11 -- Write Modelfile & Register with Ollama

Gemma 4 uses `<start_of_turn>` / `<end_of_turn>` chat tokens.

> **VERIFY:** Confirm the TEMPLATE syntax against the Ollama Gemma library page
> before running: https://ollama.com/library/gemma3 (use equivalent for Gemma 4)


In [ ]:
import subprocess
from pathlib import Path
GGUF_FILE = Q8_GGUF
if not GGUF_FILE.exists():
    raise FileNotFoundError(f'{GGUF_FILE} not found -- run Cell 8b first.')

print(f'Using GGUF: {GGUF_FILE}  ({GGUF_FILE.stat().st_size / 1e9:.1f} GB)')

SYSTEM_PROMPT = (
    'You are ARIA, an AML (Anti-Money Laundering) analytics AI assistant built by Xceed. '
    'You analyze false positive/false negative trade-offs in AML alert thresholds, '
    'perform customer behavioral segmentation, and interpret clustering results. '
    'Use the available tools to retrieve data, then provide clear, analytical insights. '
    'IMPORTANT: You MUST respond entirely in English. '
    'Be concise and reference specific numbers when interpreting results.'
)

lines = [
    f'FROM {GGUF_FILE}',
    '',
    'PARAMETER num_ctx 8192',
    'PARAMETER temperature 0.1',
    'PARAMETER top_p 0.9',
    'PARAMETER stop <turn|>',
    'PARAMETER stop <eos>',
    '',
    'TEMPLATE """',
    '{{ if .System }}<|turn>user',
    '{{ .System }}<turn|>',
    '{{ end }}',
    '{{ range .Messages }}',
    '{{ if eq .Role "user" }}<|turn>user',
    '{{ .Content }}<turn|>',
    '<|turn>model',
    '{{ else if eq .Role "assistant" }}{{ .Content }}<turn|>',
    '{{ else if eq .Role "tool" }}<|turn>tool',
    '{{ .Content }}<turn|>',
    '<|turn>model',
    '{{ end }}',
    '{{ end }}',
    '"""',
    '',
    f'SYSTEM "{SYSTEM_PROMPT}"',
]

modelfile_content = chr(10).join(lines)
modelfile_path = Path('/tmp/Modelfile.aml')
modelfile_path.write_text(modelfile_content, encoding='utf-8')
print('--- Modelfile ---')
print(modelfile_content)

## Cell 12 -- Start Ollama Server

Binds to 0.0.0.0:11434. Expose via vast.ai port forwarding or Cloudflare tunnel.


In [ ]:
import time

subprocess.run("pkill -f 'ollama serve' || true", shell=True)
time.sleep(2)
env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'
proc = subprocess.Popen(['ollama', 'serve'], env=env,
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print(f'Ollama server PID: {proc.pid}')
r = subprocess.run(['curl', '-s', 'http://localhost:11434/api/tags'],
                   capture_output=True, text=True)
print('Health check:', r.stdout[:200])


In [ ]:
print(f'Registering {OLLAMA_MODEL_NAME} ...')
result = subprocess.run(
    ['ollama', 'create', OLLAMA_MODEL_NAME, '-f', str(modelfile_path)],
    capture_output=True, text=True
)
print(result.stdout[-500:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
    raise RuntimeError('ollama create failed')
r = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
print('Registered models:')
print(r.stdout)


## Cell 14 -- Smoke Test

In [ ]:
import urllib.request, json as _json

payload = _json.dumps({
    'model': OLLAMA_MODEL_NAME,
    'messages': [{'role': 'user', 'content': 'Show FP/FN trade-off for Business customers by monthly transaction amount'}],
    'stream': False,
}).encode()
req = urllib.request.Request(
    'http://localhost:11434/api/chat',
    data=payload,
    headers={'Content-Type': 'application/json'},
)
with urllib.request.urlopen(req, timeout=120) as resp:
    result = _json.loads(resp.read())
print('Response:')
print(result['message']['content'][:800])


## Cell 15 -- Connect Local Dash App

On your **local Windows machine**:

```powershell
$env:OLLAMA_BASE_URL = "http://<vast-ai-ip>:11434/v1"
$env:OLLAMA_MODEL    = "aria-v2"
.venv\Scripts\python.exe application.py
```

Or use a Cloudflare tunnel on the vast.ai instance:
```bash
cloudflared tunnel --url http://localhost:11434
```
